# Figure 3, Panel G

Heatmaps of NES and adjusted pval from GSEA comparing BAFinh and Veh at 1h, 6h, and 30h for top 15 GO:BP terms with most negative NES. Ordering is based on rowMax NES and is the same for both heatmaps

*Output: PanelG.pdf (NES); PanelG_p.pdf (padj)*

# Setup

In [1]:
# load packages
library(tidyverse)
library(fgsea)
library(msigdbr)
library(patchwork)
library(RColorBrewer)
library(rtracklayer)
library(pheatmap)
library(here)
here::i_am("figures/Figure3/PanelG/PanelG.ipynb")

theme_set(theme_bw() + theme(text=element_text(size=15), panel.grid=element_blank()))

#function to perform GSEA analysis and annotate results based on whether GO:BP term is of type STIMULUS or RESPONSE
gsea <- function(table, genesets){
    set.seed(1)
    ranking.vals <- table$log2FoldChange
    names(ranking.vals) <- table$gene

    res <- fgsea(pathways=genesets, stats=ranking.vals,minSize=10)
    res$isStim <- grepl("STIMULUS|RESPONSE", res$pathway)
    return(res)
}
#function to deal with duplicate gene names
collapse_deg_duplicates <- function(df,
                                    gene_col = "gene",
                                    padj_col = "padj",
                                    lfc_col  = "log2FoldChange") {
  df %>%
    filter(!is.na(.data[[gene_col]])) %>%
    group_by(.data[[gene_col]]) %>%
    arrange(.data[[padj_col]], desc(abs(.data[[lfc_col]]))) %>%
    dplyr::slice(1) %>%
    ungroup()
}

sessionInfo()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:lubridate’:

    intersect, setdiff, union


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:b

R version 4.2.2 Patched (2022-11-10 r83330)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 23.04

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] here_1.0.2           pheatmap_1.0.12      rtracklayer_1.58.0  
 [4] GenomicRanges_1.50.2 GenomeInfoDb_1.34.9  IRanges_2.32.0      
 [7] S4Vectors_0.36.2     BiocGenerics_0.44.0  RColorBrewer_1.1-3  
[10] patchwork_1.3.0      msigdbr_7.5.1        fgsea_1.24.0        
[13] lub

# Load GO:BP terms from msigdb

In [2]:
#load GO:BP terms from msigdbr
all_gene_sets = msigdbr(species = "Homo sapiens")
keeper.cats <- c("GO:BP")
all_gene_sets <- all_gene_sets[which(all_gene_sets$gs_subcat %in% keeper.cats),]

#convert terms to fgsea_compatible format
msigdbr_list = split(x = all_gene_sets$gene_symbol, f = all_gene_sets$gs_name)

# Load data

In [3]:
#load DEG datasets
brm_1h <- read.delim(
  here(
    "inputs", "DEGs", "GM12878", "BAFinh_stimulus", "4rep", 
    "BAFinh_1h_DEGs.tsv"
  ),
  sep = "\t"
) %>% collapse_deg_duplicates()

brm_6h <- read.delim(
  here(
    "inputs", "DEGs", "GM12878", "BAFinh_stimulus", "4rep", 
    "BAFinh_6h_DEGs.tsv"
  ),
  sep = "\t"
) %>% collapse_deg_duplicates()

brm_30h <- read.delim(
  here(
    "inputs", "DEGs", "GM12878",
    "BRM_30h_DEGs.tsv"
  ),
  sep = "\t"
) %>% collapse_deg_duplicates()

# Perform GSEA

In [4]:
#run GSEA
brm_1h_gsea <- gsea(brm_1h,genesets=msigdbr_list)
brm_6h_gsea <- gsea(brm_6h, genesets=msigdbr_list)
brm_30h_gsea <- gsea(brm_30h, genesets=msigdbr_list)

Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“There were 10 pathways for which P-values were not calculated properly due to unbalanced (positive and negative) gene-level statistic values. For such pathways pval, padj, NES, log2err are set to NA. You can try to increase the value of the argument nPermSimple (for example set it nPermSimple = 10000)”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“For some of the pathways the P-values were likely overestimated. For such pathways log2err is set to NA.”
Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.04% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


# make data frames of NES and adjusted pvals across timepoints

In [5]:
brm_1h_slim <- brm_1h_gsea %>% select(pathway, NES)
brm_6h_slim <- brm_6h_gsea %>% select(pathway, NES)
brm_30h_slim <- brm_30h_gsea %>% select(pathway, NES)

all_nes <- inner_join(inner_join(brm_1h_slim, brm_6h_slim, by="pathway"), brm_30h_slim, by="pathway") %>%
    `colnames<-`(c("pathway", "NES_1h", "NES_6h", "NES_30h")) 

brm_1h_slim <- brm_1h_gsea %>% select(pathway, padj)
brm_6h_slim <- brm_6h_gsea %>% select(pathway, padj)
brm_30h_slim <- brm_30h_gsea %>% select(pathway, padj)

all_padj <- inner_join(inner_join(brm_1h_slim, brm_6h_slim, by="pathway"), brm_30h_slim, by="pathway") %>%
    `colnames<-`(c("pathway", "padj_1h", "padj_6h", "padj_30h")) 

# Generate plots

In [11]:
#order rows by rowMax NES and select 15 NES with most negative max NES scores
order <- (all_nes %>% column_to_rownames("pathway") %>%
    mutate(id=row_number(), max_NES=apply(all_nes %>% select(!c(pathway)), 1, min)) %>%
    arrange(max_NES) %>%
    head(15))$id

#plot NES
all_nes[order,] %>% column_to_rownames("pathway") %>%
    pheatmap(cluster_rows=F, cluster_cols=F, 
             breaks=seq(-3,0, length.out=100),
             color=colorRampPalette(c("black", "white"))(100), filename="PanelG.pdf",  cellheight=10, cellwidth=10)
#plot adjusted pvals
all_padj[order,] %>% column_to_rownames("pathway") %>%
    apply(2, function(x) -1*log10(x)) %>%
    pheatmap(cluster_rows=F, cluster_cols=F, 
             breaks=seq(0,20, length.out=100),
             color=colorRampPalette(c("white", "navy"))(100), filename="PanelG_p.pdf",cellheight=10, cellwidth=10)